# Hi! 👋

The following notebook is used to **visualize a decentralized experiments**. If you are running the following notebook in Colab, the next cell will ask you for a GDrive URL and a github token. The token will be used to clone [DecentralizedLearning/dEngine](https://github.com/DecentralizedLearning/dEngine). If you don't own the repo, the token can be generated using Github web (`settings > Developer Settings > Personal Access Tokens`). The GDrive URL will be provided by who ran the experiments. 

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import gdown

try:  # Check if we are in Google colab
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
    
    !git clone --quiet https://github.com/DecentralizedLearning/dEngine ..
    !pip install --quiet git+https://github.com/DecentralizedLearning/notebooks.git
    
    EXPERIMENT_GDRIVE_URL = input('Experiment remote URL \t')
    LOGS_PATH = Path('/content/logs')
    if not os.path.exists(LOGS_PATH):
        gdown.download_folder(url=EXPERIMENT_GDRIVE_URL, output=str(LOGS_PATH))
        !unzip -q {LOGS_PATH}/\*.zip -d {LOGS_PATH}    
except:
    pass

<br><br>
<hr>

<br><br>

# Experiment Selection

The next cell will open a file selection widget.

Please select the directory containing the experiment logs (usually inside a `log/` folder).

**Note**: If you are using Google Drive, you may need to navigate one level up. Logs are typically downloaded to `/content/logs`, while the code runs in `/content/SaiSim`.

In [ ]:
from dnotebooks.widgets import MultiExperimentSelection

experiment_selection_widget, get_selected_experiments = MultiExperimentSelection(limit=2)
display(experiment_selection_widget)

<br>
The next cell will open a dropdown selection widget.

Please select a confusion matrix file. These files can be found in `<selected_experiment>/metrics/<node_id>/` and they contain NumPy arrays generated by the simulation callbacks.

You can also load them manually with NumPy:
```
import numpy as np
from saisim.config.constants import METRICS_DIR_NAME

experiment_root, linestyle = get_selected_experiments()[0]
np.load((experiment_root / METRICS_DIR_NAME) / "<node_id>/<filename>.npy")
```

In [ ]:
from dnotebooks.widgets import ConfusionMatrixPartitionDeltaSelection

selected_experiments = get_selected_experiments()
confusion_matrix_partition_selection_wg, get_confusion_matrix = ConfusionMatrixPartitionDeltaSelection(selected_experiments)
confusion_matrix_partition_selection_wg

<br><br>
<hr>
<br><br>

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [8, 5]

from IPython.display import display

from dengine.analysis import Experiment
from dnotebooks.widgets import ExperimentDashboard
from dnotebooks.plots import plot_partition_distribution, plot_scatterplot_partition_distribution

In [ ]:
selected_experiment_path = get_selected_experiments()[0]
experiment = Experiment(
    experiment_root_path=selected_experiment_path, 
    dataset_root_path=Path("./datasets/")
)

try:
    confusion_matrices = get_confusion_matrix()
except:
    confusion_matrices = None

In [ ]:
%matplotlib widget

plt.close('all') 

data_distr_fig, data_distr_ax = plt.subplots(figsize=(10, 5))
plot_scatterplot_partition_distribution(experiment.partitions, data_distr_ax)
data_distr_fig.tight_layout()
plt.show()

dashboard = ExperimentDashboard(
    selected_experiment_path, 
    confusion_matrices, 
    experiment.partitions,
    experiment.training_engine.graph,
)
display(dashboard.render())

<br><br>
<hr>
<br><br>